# MOMCare: Data Preprocessing and Metadata Addition (Zarak)


In [ ]:
# Install dependencies
!pip install -q beautifulsoup4

In [ ]:
!pip install -q sentence-transformers

In [ ]:
# Imports
import pandas as pd
import re
from bs4 import BeautifulSoup
from google.colab import files
import io
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split

In [ ]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Extended stopwords for clinical context
clinical_stopwords = set([
    "patient", "doctor", "nurse", "hospital", "clinic", "healthcare",
    "symptom", "treatment", "medical", "diagnosis", "disease", "illness"
])
stop_words.update(clinical_stopwords)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# Upload files
uploaded = files.upload()

print(uploaded.keys()) # check directory for uploaded files

Saving 4_Motivational_Dataset.csv to 4_Motivational_Dataset.csv
Saving 2_Edge_Cases Dataset.csv to 2_Edge_Cases Dataset.csv
Saving 3_Counsel_Chat_Dataset.csv to 3_Counsel_Chat_Dataset.csv
Saving 1_PPD_Dataset.xlsx to 1_PPD_Dataset.xlsx
dict_keys(['4_Motivational_Dataset.csv', '2_Edge_Cases Dataset.csv', '3_Counsel_Chat_Dataset.csv', '1_PPD_Dataset.xlsx'])


In [ ]:
print(uploaded.keys())


dict_keys(['4_Motivational_Dataset.csv', '2_Edge_Cases Dataset.csv', '3_Counsel_Chat_Dataset.csv', '1_PPD_Dataset.xlsx'])


In [ ]:
# Load datasets
ppd_df = pd.read_excel(io.BytesIO(uploaded['1_PPD_Dataset.xlsx']))
edge_df = pd.read_csv(io.BytesIO(uploaded['2_Edge_Cases Dataset.csv']))
counsel_df = pd.read_csv(io.BytesIO(uploaded['3_Counsel_Chat_Dataset.csv']))
motiv_df = pd.read_csv(io.BytesIO(uploaded['4_Motivational_Dataset.csv']), encoding='latin1')


#  Add Domain Labels
ppd_df["Domain"] = "PPD"
edge_df["Domain"] = "Edge_Cases"
counsel_df["Domain"] = "Counsel_Chat"
motiv_df["Domain"] = "Motivational"

In [ ]:
# Show 5 random rows from each dataset
print("PPD Dataset Sample:")
display(ppd_df.sample(5))

print("Edge Cases Dataset Sample:")
display(edge_df.sample(5))

print("Counsel Chat Dataset Sample:")
display(counsel_df.sample(5))

print("Motivational Dataset Sample:")
display(motiv_df.sample(5))


PPD Dataset Sample:


,Question,Response,Domain
436,I donâ€™t have a close relationship with my fa...,"There are various avenues for support, such as...",PPD
253,Why am I feeling this way now when I didn’t ha...,It’s completely understandable to feel confuse...,PPD
145,I feel like I’ve lost myself Who am I even any...,Feeling like you’ve lost yourself is a profoun...,PPD
8,Nobody offered help that actually made a diffe...,"The well-intentioned but unhelpful ""let me kno...",PPD
77,"I don’t want big, grand gestures. I just want ...","The desire for quiet noticing, for someone to ...",PPD


Edge Cases Dataset Sample:


,Question,Response,Domain
0,lost,Feeling lost can be really difficult. I’m here...,Edge_Cases
1,help me,"Of course, I’m here for you. What do you feel ...",Edge_Cases
89,"I feel like nobody cares about me, but I don’t...",It sounds like you’re feeling isolated but als...,Edge_Cases
25,"I feel overwhelmed, I’m not happy with my appe...",It sounds like you’re feeling self-critical an...,Edge_Cases
55,I don’t see how things can ever get better.,"When things feel this dark, it’s hard to imagi...",Edge_Cases


Counsel Chat Dataset Sample:


,Question,Response,Domain
107,I was born a girl. I look like a boy. I someti...,<p>It is ok to tell someone who is casually as...,Counsel_Chat
45,Me and my adult daughter just do not get along...,<p>As frustrating and probably hurtful as your...,Counsel_Chat
1,My dad passed away when I was a teenager. I ne...,<p>It's never to late to get help with grief.&...,Counsel_Chat
106,It's really hard to not have negative feelings...,"<p>One thing I would ask is ""why are you still...",Counsel_Chat
75,Is this something I should be worried about? ...,"<ul style=""margin-right: 0px; margin-bottom: 0...",Counsel_Chat


Motivational Dataset Sample:


,Question,Response,Domain
43,i feel anxious a lot. could picking up a hobby...,engaging in a hobby can help manage anxiety by...,Motivational
99,i want to improve my communication at work. ar...,many personal growth workshops cover effective...,Motivational
87,i'm in recovery and sometimes struggle with se...,daily affirmations can reinforce positivity. t...,Motivational
58,my workplace relationships feel strained. coul...,"exercise can relieve tension and improve mood,...",Motivational
11,anxiety makes me feel like i can't handle life...,hearing others' stories of resilience can be i...,Motivational


In [ ]:
print(ppd_df.columns.tolist())
print(edge_df.columns.tolist())
print(counsel_df.columns.tolist())
print(motiv_df.columns.tolist())

['Question', 'Response', 'Domain']
['Question', 'Response', 'Domain']
['Question', 'Response', 'Domain']
['Question', 'Response', 'Domain']


In [ ]:
'''# Strip HTML from Response in counsel_df
def strip_html(text):
    if pd.isna(text): #(I added this but this didnt work)
        return text  # leave NaN as is (NEW ADD)
    return BeautifulSoup(text, "html.parser").get_text()


counsel_df["Response"] = counsel_df["Response"].apply(strip_html)

print("Counsel Chat Dataset (Cleaned):")
display(counsel_df.sample(5))'''

'# Strip HTML from Response in counsel_df\ndef strip_html(text):\n    if pd.isna(text): #(I added this but this didnt work)\n        return text  # leave NaN as is (NEW ADD)\n    return BeautifulSoup(text, "html.parser").get_text()\n\n\ncounsel_df["Response"] = counsel_df["Response"].apply(strip_html)\n\nprint("Counsel Chat Dataset (Cleaned):")\ndisplay(counsel_df.sample(5))'

In [ ]:
import unicodedata

def clean_text(text):
    text = unicodedata.normalize("NFKD", str(text))
    text = text.encode("ascii", "ignore").decode("ascii")
    return text.strip()

# Applying cleaning to each dataset
for df_clean in [ppd_df, edge_df, counsel_df, motiv_df]:
    df_clean["Question"] = df_clean["Question"].apply(clean_text)
    df_clean["Response"] = df_clean["Response"].apply(clean_text)


In [ ]:
# Remove rows with missing values
for df in [ppd_df, edge_df, counsel_df, motiv_df]:
    df.dropna(subset=["Question", "Response"], inplace=True)

# Remove duplicates
for df in [ppd_df, edge_df, counsel_df, motiv_df]:
    df.drop_duplicates(subset=["Question", "Response"], inplace=True)

In [ ]:
# Strip HTML from Response in counsel_df
def strip_html(text):
    return BeautifulSoup(text, "html.parser").get_text()

counsel_df["Response"] = counsel_df["Response"].apply(strip_html)

print("Counsel Chat Dataset (Cleaned):")
display(counsel_df.sample(5))

Counsel Chat Dataset (Cleaned):


,Question,Response,Domain
76,I recently lost a friend to suicide. I'm smok...,"First of all, I am very sorry for your loss, a...",Counsel_Chat
114,I am pretty sure I have depression and anxiety...,Family support is very helpful when having the...,Counsel_Chat
92,I recently lost a friend to suicide. I'm smok...,I urge you to seek some therapeutic help for t...,Counsel_Chat
105,"I'm a guy. If I don't like girls, nor do I lik...","Hi, and thanks for your question. I agree with...",Counsel_Chat
45,Me and my adult daughter just do not get along...,As frustrating and probably hurtful as your da...,Counsel_Chat


In [ ]:
# Drop unwanted columns
for df in [ppd_df, edge_df, counsel_df, motiv_df]:
    for col in ["norm_q", "norm_a"]:  # Iterate through columns to drop
        if col in df.columns:  # Check if column exists in DataFrame
            df.drop(columns=[col], inplace=True)  # Drop column if present

In [ ]:
# Strip leading/trailing whitespace
for df in [ppd_df, edge_df, counsel_df, motiv_df]:
    df["Question"] = df["Question"].str.strip()
    df["Response"] = df["Response"].str.strip()

In [ ]:
# Merge all 4 cleaned datasets
df = pd.concat([ppd_df, edge_df, counsel_df, motiv_df], ignore_index=True)

In [ ]:
'''category_keywords = {
    "Mental & Physical Health Support": ["depression", "anxiety", "fatigue", "insomnia", "headache"],
    "Relationship & Social Support": ["husband", "partner", "family", "friend", "in-laws", "mother"],
    "Parenting & Baby Care Stress": ["baby", "breastfeeding", "crying", "sleep", "child", "feeding"],
    "Self-care & Coping Strategies": ["exercise", "therapy", "self-care", "routine", "relax", "cope"],
    "Crisis & Emergency Support": ["suicide", "kill", "die", "self-harm", "cutting", "jump", "emergency"],
    "Follow-up & Continuous Support": ["still feel", "again", "keep happening", "not improving"]
}'''

In [ ]:
'''# Category Assignment Function (regex based)
def assign_category(question):
    question_lower = question.lower()
    for category, keywords in category_keywords.items():
        for kw in keywords:
            if re.search(r'\b' + re.escape(kw) + r's?\b', question_lower):
                return category
    return "Uncategorized"'''



In [ ]:
!pip install --upgrade transformers --upgrade-strategy eager
!pip install --upgrade accelerate


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.4/481.4 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.7/128.7 kB 9.5 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.3.0
    Uninstalling urllib3-2.3.0:
      Successfully uninstalled urllib3-2.3.0
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.13.1
    Uninstalling typing_extensions-4.13.1:
      Successfully uninstalled typing_extensions-4.13.1
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninsta

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 6.4 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.5.2
    Uninstalling accelerate-1.5.2:
      Successfully uninstalled accelerate-1.5.2


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [ ]:
#Importing Embedding Model
# Expanded Category Keywords
category_keywords = {
    "Mental & Physical Health Support": [
        "depression", "anxiety", "fatigue", "insomnia", "headache", "panic", "overwhelmed", "mental health", "sadness", "emotions"
    ],
    "Relationship & Social Support": [
        "husband", "partner", "family", "friend", "fight", "mother", "father", "support", "communication", "lonely"
    ],
    "Parenting & Baby Care Stress": [
        "baby", "breastfeeding", "crying", "sleep", "child", "feeding", "newborn", "infant", "childbirth", "motherhood"
    ],
    "Self-care & Coping Strategies": [
        "exercise", "therapy", "self-care", "routine", "relax", "cope", "mindfulness", "journaling", "walk", "breathing"
    ],
    "Crisis & Emergency Support": [
        "suicide", "kill", "die", "self-harm", "cutting", "jump", "emergency", "harm myself", "life danger", "hopeless"
    ],
    "Follow-up & Continuous Support": [
        "still feel", "again", "keep happening", "not improving", "relapse", "feeling worse", "therapy again", "backslide", "return", "continue"
    ],
    "General Mental Health Support": [
        "anxiety", "trauma", "therapy", "counseling", "mental", "depressed", "self-esteem", "trust", "emotion"
    ],
    "Motivational/Resilience Support": [
        "affirmation", "resilience", "self-esteem", "healing", "positivity", "overcoming", "strength", "hope", "empowerment"
    ]

}

# Synthetic labeled examples
synthetic_examples = {
    "Mental & Physical Health Support": [
        "Why do I feel so empty even after giving birth?",
        "I'm anxious all the time about everything.",
        "Sleep feels impossible lately, what can I do?",
        "I constantly feel tired, even with rest.",
        "Sometimes I get headaches after feeding my baby.",
        "Is it normal to feel emotionally numb postpartum?",
        "I'm feeling very panicked these days.",
        "My emotions are all over the place.",
        "I feel a deep sadness that won't go away.",
        "I have trouble managing my stress after delivery."
    ],
    "Relationship & Social Support": [
        "My husband doesn’t understand my feelings after birth.",
        "I feel distant from my family since I became a mother.",
        "My friends have disappeared since I had the baby.",
        "I feel lonely even when people are around me.",
        "My partner and I argue all the time now.",
        "How do I talk to my mother-in-law about my needs?",
        "I feel unsupported by my loved ones.",
        "Why do I feel isolated from my friends after delivery?",
        "I need better communication with my spouse.",
        "How can I rebuild my support system?"
    ],
    "Parenting & Baby Care Stress": [
        "Why won't my baby stop crying?",
        "I'm so tired of changing diapers all day.",
        "Breastfeeding is so painful for me.",
        "My baby never sleeps through the night.",
        "I can't soothe my infant when she's fussy.",
        "Feeding the baby feels like a huge stressor.",
        "How do I deal with newborn sleep issues?",
        "My child refuses to eat, and I'm worried.",
        "Being a mother is harder than I thought.",
        "Colic is driving me crazy, help!"
    ],
    "Self-care & Coping Strategies": [
        "How can I find time for myself with a newborn?",
        "Are there exercises safe for postpartum moms?",
        "Would journaling help my stress?",
        "What are some quick mindfulness exercises?",
        "Is therapy recommended for new moms?",
        "I need a daily routine to feel stable.",
        "Breathing exercises for calming down?",
        "What small self-care routines can I do?",
        "How can I relax when I feel so busy?",
        "Tips to cope with feeling overwhelmed?"
    ],
    "Crisis & Emergency Support": [
        "Sometimes I think about ending it all.",
        "I feel like hurting myself, what should I do?",
        "Is there a crisis hotline I can call?",
        "I’m scared I might harm myself.",
        "Emergency help for postpartum depression?",
        "I feel unsafe being alone right now.",
        "What to do if I feel like jumping off?",
        "I'm scared I might hurt the baby or myself.",
        "How do I get immediate mental health support?",
        "I feel completely hopeless."
    ],
    "Follow-up & Continuous Support": [
        "My depression is returning after feeling better.",
        "What if therapy doesn't help anymore?",
        "Why am I feeling bad again after recovery?",
        "I keep sliding back into sadness.",
        "Is it normal to need ongoing therapy?",
        "How to handle relapse after postpartum depression?",
        "My emotions are worsening again, help.",
        "Continuous support for long-term postpartum care?",
        "I feel like I'm back to square one.",
        "Is it bad if postpartum depression symptoms return?"
    ],
    "General Mental Health Support": [
        "I have trouble trusting people after trauma.",
        "Why do I feel depressed even when things seem fine?",
        "Therapy seems scary, how do I start?",
        "I feel stuck emotionally, what can I do?",
        "How to rebuild self-esteem after a breakup?",
        "Why do I push people away emotionally?",
        "How can therapy help with emotional issues?",
        "I want to find emotional balance again.",
        "Anxiety makes daily tasks impossible for me.",
        "I feel misunderstood by everyone."
    ],
    "Motivational/Resilience Support": [
        "How can I stay positive during hard times?",
        "What affirmations help boost self-confidence?",
        "Stories of people overcoming depression?",
        "Ways to build resilience after trauma?",
        "I want to feel strong again.",
        "How can I cultivate hope daily?",
        "Motivational quotes for healing?",
        "Activities that build emotional strength?",
        "Tips for bouncing back from failure?",
        "I want to feel empowered again."
    ]
}

# Embedding Model (clinical fine-tuned not mini lm)

embed_model = SentenceTransformer('neuml/pubmedbert-base-embeddings')

#embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
#embed_model = SentenceTransformer('pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb')

# Preparing synthetic embeddings
synthetic_questions = []
synthetic_labels = []
for label, examples in synthetic_examples.items():
    synthetic_questions.extend(examples)
    synthetic_labels.extend([label] * len(examples))

synthetic_embeddings = embed_model.encode(synthetic_questions)

# Embedding-Based Assignment
def assign_by_embedding(question):
    question_embed = embed_model.encode([question])[0]
    sims = cosine_similarity([question_embed], synthetic_embeddings)[0]
    best_idx = np.argmax(sims)
    best_score = sims[best_idx]
    best_label = synthetic_labels[best_idx]

    if best_score > 0.65:  # Threshold setting to 0.65 simil
        return best_label
    else:
        return "Uncategorized"

# Final Hybrid Category Assignment
def assign_category_hybrid(question):
    # Trying keyword match first
    cat = None
    question_lower = question.lower()
    for category, keywords in category_keywords.items():
        for kw in keywords:
            if re.search(r'\b' + re.escape(kw) + r's?\b', question_lower):
                cat = category
                break
        if cat:
            break

    if cat:
        return cat
    else:
        return assign_by_embedding(question)



In [ ]:
# Urgency Flag Function
def check_urgency(text):
    crisis_words = ["suicide", "kill myself", "self-harm", "die", "jump", "cutting"]
    return any(word in text.lower() for word in crisis_words)

In [ ]:
# Keyword Tag Extractor (nltk based)
def extract_keywords(text):
    words = re.findall(r'\b\w+\b', str(text).lower())
    keywords = [word for word in words if word not in stop_words and len(word) > 3]
    return ", ".join(keywords[:5])

In [ ]:
# Adding Metadata Columns
df['question_length'] = df['Question'].str.split().apply(len)
df['answer_length'] = df['Response'].astype(str).str.split().apply(len)
df['Category'] = df['Question'].apply(assign_category_hybrid)
df['is_urgent'] = df['Question'].apply(check_urgency)
df['keyword_tags'] = df['Question'].apply(extract_keywords)

In [ ]:
print(df.shape)
display(df.sample(5))

(1003, 8)


,Question,Response,Domain,question_length,answer_length,Category,is_urgent,keyword_tags
633,Can you recommend a movie to watch?,Movies can be a great escape! I focus on emoti...,Edge_Cases,7,22,Uncategorized,False,"recommend, movie, watch"
66,"I want to feel like were connected, like were ...",Feeling connected amidst the chaos involves co...,PPD,24,34,Uncategorized,False,"want, feel, like, connected, like"
32,Do other couples go through this awkward thing...,Navigating physical closeness postpartum is of...,PPD,29,41,Mental & Physical Health Support,False,"couples, awkward, thing, physical, closeness"
462,What are some ways to relieve anxiety about my...,"It can be distressing, but remember that babie...",PPD,11,44,Mental & Physical Health Support,False,"ways, relieve, anxiety, babyaaas, crying"
109,Will I get PPD again if I have another baby? W...,Having had postpartum depression (PPD) once do...,PPD,27,79,Parenting & Baby Care Stress,False,"another, baby, pregnancy, lower, risk"


In [ ]:
'''# Split dataset into training (80%) and testing (20%)
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)

print("Training Data Size:", train_df.shape)
print("Testing Data Size:", test_df.shape)'''

Training Data Size: (1646, 8)
Testing Data Size: (183, 8)


In [ ]:
# Save final enriched dataset
df.to_csv("embedded_dataset.csv", index=False)
print("Metadata added successfully! The dataset is saved as 'embedded_dataset.csv'")


Metadata added successfully! The dataset is saved as 'embedded_dataset.csv'


In [ ]:
# Download the final file
files.download("embedded_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>